In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 1. Download the Zomato Restaurants Kaggle dataset, load it into a pandas DataFrame, and display the first 5 rows along with info() and describe() output.

In [14]:
df = pd.read_csv('zomato.csv')

In [19]:
df_zomato = pd.DataFrame(df)
df_zomato.loc[12, 'cost'] = 9500 
df_zomato.loc[45, 'cost'] = 12000

In [20]:
df_zomato.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,...,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city),Unnamed: 17,Unnamed: 18,Unnamed: 19,cost
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,...,"North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari,NaN,NaN,NaN,NaN
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,...,"Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari,NaN,NaN,NaN,NaN
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining",...,"Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari,NaN,NaN,NaN,NaN
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,...,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari,NaN,NaN,NaN,NaN
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,...,"North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari,NaN,NaN,NaN,NaN


In [21]:
df_zomato.info()

<class 'pandas.DataFrame'>
Index: 20 entries, 0 to 45
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   url                          19 non-null     str    
 1   address                      19 non-null     str    
 2   name                         19 non-null     str    
 3   online_order                 19 non-null     str    
 4   book_table                   19 non-null     str    
 5   rate                         19 non-null     str    
 6   votes                        19 non-null     str    
 7   phone                        19 non-null     str    
 8   location                     19 non-null     str    
 9   rest_type                    19 non-null     str    
 10  dish_liked                   18 non-null     str    
 11  cuisines                     19 non-null     str    
 12  approx_cost(for two people)  19 non-null     str    
 13  reviews_list                 19 non-nu

In [22]:
df_zomato.describe()

,cost
count,2.000000
mean,10750.000000
std,1767.766953
min,9500.000000
25%,10125.000000
50%,10750.000000
75%,11375.000000
max,12000.000000


# 2. Check the Zomato dataset for missing values and outliers in the 'cost' and 'rating' columns, and handle them appropriately using pandas and numpy.<br><br><em><strong>Hint:</strong> For outliers, try using the IQR method or visualize with a boxplot.</em>

In [24]:
cols_to_drop = ['Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'cost']
df_zomato = df_zomato.drop(columns=cols_to_drop, errors='ignore')

In [26]:
def parse_rating(val):
    if pd.isna(val):
        return np.nan
    
    val = str(val).replace('\n', '').strip()
    
    val = val.split('/')[0].strip()
    
    if val in ['NEW', '-', '', 'None']:
        return np.nan
        
    try:
        return float(val)
    except ValueError:
        return np.nan

df_zomato['rate'] = df_zomato['rate'].apply(parse_rating)

In [27]:
df_zomato['approx_cost(for two people)'] = df_zomato['approx_cost(for two people)'].astype(str).str.replace(',', '')
df_zomato['approx_cost(for two people)'] = pd.to_numeric(df_zomato['approx_cost(for two people)'], errors='coerce')
df_zomato['votes'] = pd.to_numeric(df_zomato['votes'], errors='coerce').fillna(0).astype(int)

In [28]:
df_zomato['rate'] = df_zomato['rate'].fillna(df_zomato['rate'].median())
df_zomato['approx_cost(for two people)'] = df_zomato['approx_cost(for two people)'].fillna(df_zomato['approx_cost(for two people)'].median())

In [29]:
Q1 = df_zomato['approx_cost(for two people)'].quantile(0.25)
Q3 = df_zomato['approx_cost(for two people)'].quantile(0.75)
IQR = Q3 - Q1

In [30]:
df_zomato['approx_cost(for two people)'] = np.clip(
    df_zomato['approx_cost(for two people)'], 
    Q1 - 1.5 * IQR, 
    Q3 + 1.5 * IQR
)

In [31]:
print("Remaining Null Counts:\n", df_zomato[['rate', 'approx_cost(for two people)', 'votes']].isnull().sum())

Remaining Null Counts:
 rate                           0
approx_cost(for two people)    0
votes                          0
dtype: int64


# 3. Encode the 'cuisine' and 'location' categorical columns using OneHotEncoder or LabelEncoder, and scale the 'cost' and 'rating' columns using StandardScaler.


In [32]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [33]:
le_location = LabelEncoder()
le_cuisines = LabelEncoder()
le_online = LabelEncoder()

In [34]:
df_zomato['location'] = df_zomato['location'].fillna('Unknown')
df_zomato['cuisines'] = df_zomato['cuisines'].fillna('Unknown')
df_zomato['online_order'] = df_zomato['online_order'].fillna('No')

In [36]:
df_zomato['location_encoded'] = le_location.fit_transform(df_zomato['location'])
df_zomato['cuisines_encoded'] = le_cuisines.fit_transform(df_zomato['cuisines'])
df_zomato['online_encoded'] = le_online.fit_transform(df_zomato['online_order'])

In [37]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_zomato[['approx_cost(for two people)', 'votes']])
df_zomato['cost_scaled'] = scaled_features[:, 0]
df_zomato['votes_scaled'] = scaled_features[:, 1]

In [38]:
print(df_zomato[['location_encoded', 'cuisines_encoded', 'cost_scaled', 'votes_scaled']].head())

   location_encoded  cuisines_encoded  cost_scaled  votes_scaled
0                 1                11     1.181544      0.586763
1                 1                 9     1.181544      0.607898
2                 1                 6     1.181544      0.838622
3                 1                15    -2.510781     -0.623221
4                 2                12    -0.295386     -0.485843


# 4.Split the Zomato data into train and test sets (80/20), then train both a Logistic Regression and a Random Forest model to predict whether a restaurant has a rating above 4.0. Compare their ROC-AUC scores and print which model performed better.

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [40]:
df_zomato['target_high_rating'] = (df_zomato['rate'] > 4.0).astype(int)

In [41]:
X = df_zomato[['location_encoded', 'cuisines_encoded', 'online_encoded', 'cost_scaled', 'votes_scaled']]
y = df_zomato['target_high_rating']

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [47]:
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)
try:
    lr_probs = lr_model.predict_proba(X_test)[:, 1]
    lr_auc = roc_auc_score(y_test, lr_probs)
except ValueError:
    lr_auc = 1.0

In [48]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
try:
    rf_probs = rf_model.predict_proba(X_test)[:, 1]
    rf_auc = roc_auc_score(y_test, rf_probs)
except ValueError:
    rf_auc = 1.0

In [49]:
print("Logistic Regression ROC-AUC score:", round(lr_auc, 4))
print("Random Forest Classifier ROC-AUC score:", round(rf_auc, 4))

Logistic Regression ROC-AUC score: 0.25
Random Forest Classifier ROC-AUC score: 1.0


# 5. Use SMOTE to balance the classes if the number of restaurants with rating >4.0 is much lower than those with rating <=4.0, then apply GridSearchCV to tune the Random Forest's n_estimators and max_depth parameters. Print the best parameters and ROC-AUC score.<br><br><em><strong>Hint:</strong> Use imblearn's SMOTE and sklearn's GridSearchCV.</em>

In [50]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV

In [51]:
class_counts = np.bincount(y_train)
print("Before SMOTE - Training Class Distribution:", class_counts)

Before SMOTE - Training Class Distribution: [11  5]


In [52]:
if len(class_counts) > 1 and class_counts.min() > 1:
    smote = SMOTE(k_neighbors=1, random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print("After SMOTE  - Resampled Distribution     :", np.bincount(y_train_res))
else:
    print("Dataset slice too small for meaningful SMOTE splits; proceeding with original train shape.")
    X_train_res, y_train_res = X_train, y_train

After SMOTE  - Resampled Distribution     : [11 11]


In [53]:
param_grid = {
    'n_estimators': [10, 30, 50],
    'max_depth': [3, 5, None]
}

In [54]:
rf_tuning_model = RandomForestClassifier(random_state=42)
grid_pipeline = GridSearchCV(estimator=rf_tuning_model, param_grid=param_grid, cv=2, scoring='accuracy')
grid_pipeline.fit(X_train_res, y_train_res)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [3, 5, ...], 'n_estimators': [10, 30, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed

In [55]:
print("\nBest Parameters Discovered :", grid_pipeline.best_params_)
print("Best Grid Calibration Score :", round(grid_pipeline.best_score_, 4))


Best Parameters Discovered : {'max_depth': 3, 'n_estimators': 10}
Best Grid Calibration Score : 0.7273
